# Calibration Status to Alarm Automation

This notebook monitors asset calibration status and automatically creates or updates alarms in SystemLink based on the calibration state of managed assets.

## Severity Mapping

The notebook maps asset calibration statuses to alarm severity levels:

- **PAST_RECOMMENDED_DUE_DATE** → Severity 4 (Critical)  
  Asset is past its recommended calibration due date - creates or updates alarm with critical severity

- **APPROACHING_RECOMMENDED_DUE_DATE** → Severity 2 (Warning)  
  Asset is approaching its recommended calibration due date - creates or updates alarm with warning severity

- **OK** → Severity -1 (Clear)  
  Asset calibration is up to date - clears any existing alarms

## Imports and Setup
This section imports required libraries

In [ ]:
import os
from datetime import datetime, timezone
from typing import Any, Collection, Dict, List, Optional
from urllib.parse import urljoin, urlsplit, urlunsplit

import scrapbook as sb
from nisystemlink.clients.alarm import AlarmClient
from nisystemlink.clients.alarm.models import (
    Alarm,
    CreateOrUpdateAlarmRequest,
    QueryAlarmsWithFilterRequest,
)
from nisystemlink.clients.assetmanagement import AssetManagementClient
from nisystemlink.clients.assetmanagement.models import (
    Asset,
    CalibrationStatus,
    QueryAssetsRequest,
)
from nisystemlink.clients.systems import SystemsClient
from nisystemlink.clients.systems.models import QuerySystemsRequest
from requests import Session
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configuration and Parameters

In this section, we define key parameters and configuration values that will be used throughout the notebook.

**Configuration:**
- **`workspace_ids`**: List of workspace IDs to monitor (empty list = all workspaces)
  - Example: `["workspace-123", "workspace-456"]` or `[]` for all workspaces
- **`calibration_subscribers`**: List of email addresses to receive notifications about calibration status changes
  - Example: `["user@example.com", "admin@example.com"]`

In [ ]:
workspace_ids: List[str] = []
calibration_subscribers: List[str] = []

## API Configuration and Endpoints
This section retrieves environment variables for authentication and defines the base URLs and API endpoints required for interacting with the SystemLink services.


In [ ]:
api_key = os.getenv("SYSTEMLINK_API_KEY")
http_url = os.getenv("SYSTEMLINK_HTTP_URI")

RETRIABLE_STATUS_CODES: Collection[int] = (500, 502, 503, 504)
RETRIABLE_METHODS: Collection[str] = ("HEAD", "GET", "OPTIONS", "POST", "PUT", "DELETE")

APPLY_DYNAMIC_STRATEGY_ENDPOINT = urljoin(
    http_url, "/ninotification/v1/apply-dynamic-strategy"
)

In [ ]:
# Calibration status -> desired severity mapping
CALIBRATION_STATUS_TO_SEVERITY = {
    CalibrationStatus.PAST_RECOMMENDED_DUE_DATE.value: 4,
    CalibrationStatus.APPROACHING_RECOMMENDED_DUE_DATE.value: 2,
    CalibrationStatus.OK.value: -1,
}

CALIBRATION_STATUS_TO_LABEL = {
    CalibrationStatus.PAST_RECOMMENDED_DUE_DATE.value: "calibration past due",
    CalibrationStatus.APPROACHING_RECOMMENDED_DUE_DATE.value: "calibration due soon",
    CalibrationStatus.OK.value: "calibration up to date",
}

## Session object builder

In [ ]:
def create_session(token: str) -> Session:
    """
    Create a requests Session with authentication headers and retry strategy.

    Args:
        token (str): API key for authentication

    Returns:
        Session: Configured Session object with retry strategy for HTTP/HTTPS requests
    """
    s = Session()
    s.headers.update(
        {
            "Accept": "application/json",
            "x-ni-api-key": token,
        }
    )

    retry_strategy = Retry(
        total=3,
        status_forcelist=RETRIABLE_STATUS_CODES,
        allowed_methods=RETRIABLE_METHODS,
        backoff_factor=2,
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    s.mount("http://", adapter)
    s.mount("https://", adapter)

    return s

In [ ]:
session = create_session(api_key)
asset_client = AssetManagementClient()
system_client = SystemsClient()
alarm_client = AlarmClient()

In [ ]:
def query_all_assets(filter: str = "", take: int = 100) -> List[Asset]:
    """
    Query all assets from SystemLink by paginating through all results.

    Args:
        filter (str): Filter string to apply when querying assets
        take (int): Number of assets to fetch per page

    Returns:
        List[Asset]: List of all assets matching the filter criteria
    """
    query_assets_request = QueryAssetsRequest(
        take=take,
        skip=0,
        filter=filter,
    )

    all_assets: List[Asset] = []

    while True:
        query_assets_response = asset_client.query_assets(query=query_assets_request)
        assets = query_assets_response.assets
        if not assets:
            break

        all_assets.extend(assets)
        query_assets_request.skip = (query_assets_request.skip or 0) + len(assets)

    return all_assets

In [ ]:
def query_systems(filter: str = "") -> List[Dict[str, str]]:
    """
    Query systems from SystemLink with id and alias projection.

    Args:
        filter (str): Filter string to apply when querying systems

    Returns:
        List[Dict[str, str]]: List of dictionaries containing system id and alias
    """
    request = QuerySystemsRequest(
        skip=0,
        take=1000,
        filter=filter,
        projection="new(id,alias)",
    )
    response = system_client.query_systems(request)
    return response.data

In [ ]:
def get_asset_path(asset: Asset) -> str:
    """
    Generate a unique asset path for alarm identification.

    Constructs a hierarchical path using vendor, model, and serial number.

    Args:
        asset (Asset): Asset object containing vendor, model, and serial information

    Returns:
        str: Formatted asset path string
    """
    vendor = asset.vendor_name or str(asset.vendor_number or "")
    model = asset.model_name or str(asset.model_number or "")
    serial_number = str(asset.serial_number or "")

    return f"Assets.{vendor}.{model}.{serial_number}.Calibration"

In [ ]:
def apply_dynamic_notification_strategy(
    to_addresses: Optional[List[str]] = None,
    subject: str = "subject",
    body: str = "body",
) -> Dict[str, Any]:
    """
    Send email notifications using SystemLink's notification service.

    Args:
        to_addresses (Optional[List[str]]): List of recipient email addresses
        subject (str): Email subject line
        body (str): Email body content

    Returns:
        bool: True if notification was successfully sent (status 204), otherwise False
    """
    if to_addresses is None:
        return

    payload: Dict[str, Any] = {
        "notificationStrategy": {
            "notificationConfigurations": [
                {
                    "addressGroup": {
                        "interpretingServiceName": "smtp",
                        "fields": {
                            "toAddresses": to_addresses,
                        },
                    },
                    "messageTemplate": {
                        "interpretingServiceName": "smtp",
                        "fields": {
                            "subjectTemplate": subject,
                            "bodyTemplate": body,
                        },
                    },
                }
            ]
        }
    }

    response = session.post(
        APPLY_DYNAMIC_STRATEGY_ENDPOINT,
        json=payload,
    )
    response.raise_for_status()
    print(response.status_code)
    return response.status_code == 204

In [ ]:
def build_email_body(
    http_url: str,
    assets_with_past_date: List[Asset],
    assets_with_approaching_due_date: List[Asset],
) -> str:
    """
    Build email notification body with asset calibration status information.

    Generates a formatted email body containing links to assets grouped by calibration status:
    - Assets with past calibration due date
    - Assets with approaching calibration due date

    Args:
        http_url (str): Base SystemLink HTTP URL
        assets_with_past_date (List[Asset]): List of assets past their calibration due date
        assets_with_approaching_due_date (List[Asset]): List of assets approaching their calibration due date

    Returns:
        str: Formatted email body as a string with asset links
    """
    parsed = urlsplit(http_url)
    host_without_api = parsed.netloc.replace("-api", "")
    base_url = urlunsplit((parsed.scheme, host_without_api, "", "", ""))

    def asset_link(asset: Asset) -> str:
        asset_id = asset.id
        return f"{base_url}/assets/{asset_id}"

    lines: List[str] = []

    # Assets with past date
    lines.append("Assets with past date")
    print("assets_with_past_date", len(assets_with_past_date))
    if assets_with_past_date:
        for asset in assets_with_past_date:
            link = asset_link(asset)
            if link:
                lines.append(link)
    else:
        lines.append("No new assets with past date.")

    lines.append("")  # blank line between sections

    # Assets with approaching due date
    lines.append("Assets with approaching due date")
    if assets_with_approaching_due_date:
        for asset in assets_with_approaching_due_date:
            link = asset_link(asset)
            if link:
                lines.append(link)
    else:
        lines.append("No new assets with approaching due date.")

    return "\n".join(lines)

## Alarm helpers

In [ ]:
def query_alarms(
    alarm_filter: str = "",
    take: int = 1000,
) -> List[Alarm]:
    """
    Query all alarms matching the filter by batching through all pages using continuation tokens.

    Args:
        alarm_filter (str): Filter string to apply when querying alarms
        take (int): Number of alarms to fetch per page

    Returns:
        List[Alarm]: List of all matching alarms
    """
    # First query
    request = QueryAlarmsWithFilterRequest(
        filter=alarm_filter,
        continuation_token=None,
        take=take,
        return_count=False,
    )
    response = alarm_client.query_alarms(request)

    all_alarms = list(response.alarms or [])
    continuation_token = response.continuation_token

    # Continue with remaining pages if continuation token exists
    while continuation_token:
        request = QueryAlarmsWithFilterRequest(
            filter=alarm_filter,
            continuation_token=continuation_token,
            take=take,
            return_count=False,
        )
        response = alarm_client.query_alarms(request)

        alarms = response.alarms or []
        all_alarms.extend(alarms)

        continuation_token = response.continuation_token

    return all_alarms


def get_alarm_severity(alarm: Dict[str, Any]) -> Optional[int]:
    """
    Extract severity level from an alarm dictionary.

    Args:
        alarm (Dict[str, Any]): Alarm dictionary containing severity information

    Returns:
        Optional[int]: Severity level as an integer, or None if not found
    """
    # Direct fields
    for key in ("currentSeverityLevel", "severityLevel"):
        if key in alarm:
            value = alarm[key]
            if isinstance(value, int):
                return value
            if isinstance(value, str):
                try:
                    return int(value)
                except ValueError:
                    pass

    # Nested current state
    for state_key in ("currentState", "state"):
        state = alarm.get(state_key)
        if isinstance(state, dict) and "severityLevel" in state:
            value = state["severityLevel"]
            if isinstance(value, int):
                return value
            if isinstance(value, str):
                try:
                    return int(value)
                except ValueError:
                    pass

    return None


def build_transition_payload(
    asset: Asset,
    alarm_id: str,
    target_severity: int,
    calibration_status: str,
) -> Dict[str, Any]:
    """
    Build alarm transition payload for creating or updating an alarm.

    Args:
        asset (Asset): Asset object containing asset details
        alarm_id (str): Unique identifier for the alarm
        target_severity (int): Desired severity level (-1 for clear, 2 for warning, 4 for critical)
        calibration_status (str): Current calibration status string

    Returns:
        Dict[str, Any]: Dictionary containing the complete alarm transition payload
    """
    display_name = asset.model_name or asset.serial_number or alarm_id
    now_utc = datetime.now(timezone.utc)
    iso_with_z = now_utc.replace(tzinfo=timezone.utc).isoformat().replace("+00:00", "Z")

    detail_text = (
        f"{display_name} calibration status = {calibration_status}, "
        f"severity = {target_severity} at {iso_with_z}"
    )
    transition_type = "SET"
    if target_severity == -1:
        transition_type = "CLEAR"
    # Optional: initialise these so we can use them safely later
    system_id = None
    system_alias = None

    # Only try to resolve system if location has a minionId
    location = asset.location
    if location and location.minion_id:
        minion_id = location.minion_id

        filter_str = f'id == "{minion_id}"'
        systems = query_systems(filter=filter_str)

        if systems:
            system_id = systems[0]["id"]
            system_alias = systems[0]["alias"]

    # Build properties base
    properties: Dict[str, Any] = {
        "Model name": asset.model_name,
        "Serial number": asset.serial_number,
        "Vendor name": asset.vendor_name,
    }

    # Only add minionId and system if we resolved them
    if system_id is not None and system_alias is not None:
        properties["minionId"] = system_id
        properties["system"] = system_alias

    # Build description depending on whether we have a system alias
    if system_alias:
        description = f"{display_name} {CALIBRATION_STATUS_TO_LABEL[calibration_status]} in {system_alias}"
    else:
        description = (
            f"{display_name} {CALIBRATION_STATUS_TO_LABEL[calibration_status]}"
        )

    payload: Dict[str, Any] = {
        "alarmId": alarm_id,
        "workspace": asset.workspace,
        "channel": alarm_id,
        "displayName": f"{display_name} {CALIBRATION_STATUS_TO_LABEL[calibration_status]}",
        "description": description,
        "transition": {
            "transitionType": transition_type,
            "occurredAt": iso_with_z,
            "severityLevel": target_severity,
            "value": calibration_status,
            "condition": calibration_status,
            "detailText": detail_text,
        },
        "properties": properties,
    }

    return payload


def upsert_alarms_for_assets(
    assets: List[Asset],
    target_severity: int,
    calibration_status: str,
    chunk_size: int = 500,
) -> List[Asset]:
    """
    Create or update alarms for a list of assets based on calibration status.

    Args:
        assets (List[Asset]): List of Asset objects to process
        target_severity (int): Desired severity level (-1 to clear, 2 for warning, 4 for critical)
        calibration_status (str): Calibration status string
        chunk_size (int): Number of assets to process per batch

    Returns:
        List[Asset]: List of assets that had their alarm state updated
    """
    assets_with_updated_state = []
    if not assets:
        return assets_with_updated_state

    for i in range(0, len(assets), chunk_size):
        chunk = assets[i : i + chunk_size]

        # Map asset paths -> asset
        asset_paths: List[str] = []
        asset_path_to_asset: Dict[str, Asset] = {}
        for asset in chunk:
            alarm_id = get_asset_path(asset)
            asset_paths.append(alarm_id)
            asset_path_to_asset[alarm_id] = asset

        if not asset_paths:
            continue

        alarm_filter = " or ".join(f'alarmId = "{p}"' for p in asset_paths)

        # Query existing alarms (now automatically batches through all pages)
        alarms = query_alarms(alarm_filter=alarm_filter)

        alarm_map: Dict[str, Dict[str, Any]] = {
            alarm["alarmId"]: alarm for alarm in alarms
        }

        for alarm_id, asset in asset_path_to_asset.items():
            existing_alarm = alarm_map.get(alarm_id)
            desired_severity = target_severity
            should_upsert = False

            if existing_alarm:
                current_severity = get_alarm_severity(existing_alarm)
                # If current severity is the same as desired, skip
                if current_severity == desired_severity:
                    continue
                # Update to new severity (2, 4 or -1)
                print(
                    f"Updating alarm for {alarm_id}: "
                    f"current_severity={current_severity}, target={desired_severity}"
                )
                should_upsert = True
            else:
                # No existing alarm
                if desired_severity == -1:
                    # For OK (-1) we do NOT create a new "clean" alarm - just skip
                    continue
                # For 2 / 4 we create the alarm
                print(f"Creating alarm for {alarm_id} with severity {desired_severity}")
                should_upsert = True

            if should_upsert:
                payload = build_transition_payload(
                    asset=asset,
                    alarm_id=alarm_id,
                    target_severity=desired_severity,
                    calibration_status=calibration_status,
                )
                try:
                    request = CreateOrUpdateAlarmRequest(**payload)
                    alarm_client.create_or_update_alarm(request)
                    print(
                        f"Alarm {'update' if existing_alarm else 'create'} OK for {alarm_id}"
                    )
                    assets_with_updated_state.append(asset)
                except Exception as e:
                    print(
                        f"Error {'updating' if existing_alarm else 'creating'} alarm for {alarm_id}: {e}"
                    )
    return assets_with_updated_state

## Implementation

In [ ]:
filter_str = ""
if workspace_ids:
    conditions = " OR ".join(f'workspace = "{ws}"' for ws in workspace_ids)
    filter_str = f"({conditions})"

print("Querying all assets...")
all_assets = query_all_assets(filter=filter_str)
print(f"Total assets: {len(all_assets)}")

# Divide assets into 3 categories based on calibrationStatus
assets_with_past_date: List[Asset] = []
assets_with_approaching_due_date: List[Asset] = []
calibrated_assets: List[Asset] = []

for asset in all_assets:
    status = asset.calibration_status
    if status == CalibrationStatus.PAST_RECOMMENDED_DUE_DATE.value:
        assets_with_past_date.append(asset)
    elif status == CalibrationStatus.APPROACHING_RECOMMENDED_DUE_DATE.value:
        assets_with_approaching_due_date.append(asset)
    elif status == CalibrationStatus.OK.value:
        calibrated_assets.append(asset)

sb.glue("Assets with calibration past due date", len(assets_with_past_date))
sb.glue("Assets with approaching due date", len(assets_with_approaching_due_date))
sb.glue("Calibrated assets", len(calibrated_assets))

# PAST_RECOMMENDED_DUE_DATE -> severity 4
new_assets_with_past_date = upsert_alarms_for_assets(
    assets=assets_with_past_date,
    target_severity=CALIBRATION_STATUS_TO_SEVERITY[
        CalibrationStatus.PAST_RECOMMENDED_DUE_DATE.value
    ],
    calibration_status=CalibrationStatus.PAST_RECOMMENDED_DUE_DATE.value,
)

# APPROACHING_RECOMMENDED_DUE_DATE -> severity 2
new_assets_with_approaching_due_date = upsert_alarms_for_assets(
    assets=assets_with_approaching_due_date,
    target_severity=CALIBRATION_STATUS_TO_SEVERITY[
        CalibrationStatus.APPROACHING_RECOMMENDED_DUE_DATE.value
    ],
    calibration_status=CalibrationStatus.APPROACHING_RECOMMENDED_DUE_DATE.value,
)

mail_body = build_email_body(
    http_url,
    new_assets_with_past_date,
    new_assets_with_approaching_due_date,
)

apply_dynamic_notification_strategy(
    calibration_subscribers,
    "Asset due-date alerts",
    mail_body,
)

# OK -> severity -1
upsert_alarms_for_assets(
    assets=calibrated_assets,
    target_severity=CALIBRATION_STATUS_TO_SEVERITY[CalibrationStatus.OK.value],
    calibration_status=CalibrationStatus.OK.value,
)